In [1]:
import asyncio
import logging

# This demo uses Academy to launch agents on Polaris
import academy
from academy.manager import Manager
from academy.agent import Agent, action
from academy.handle import Handle
from academy.exchange.cloud import HttpExchangeFactory
from academy.logging import init_logging

# We use Globus Compute to launch Academy Agents on Polaris
from globus_compute_sdk import Executor as GlobusComputeExecutor

print(academy.__version__)

0.4.0


In [2]:

class CandidateGenerator(Agent):
    
    def __init__(self):
        super().__init__()
        
    def _parse_json_response(self, text: str):
        """Strip markdown fences (if any) and parse JSON."""
        import json
        
        cleaned = text.strip()
        if cleaned.startswith("```"):
            # Remove opening fence (```json or ```)
            cleaned = cleaned.split("\n", 1)[-1]
        if cleaned.endswith("```"):
            cleaned = cleaned.rsplit("```", 1)[0]
        return json.loads(cleaned.strip())
        
    @action
    async def generate_candidates(self, target_description: str, n: int = 5) -> list[dict]:
        """Ask the LLM to generate candidate drug-like SMILES for a target."""

        from langchain_openai import ChatOpenAI
        from inference_auth_token import get_access_token

        # Get your access token for Argonne's Inference Endpoint
        api_key = get_access_token()

        model = ChatOpenAI(
            model='openai/gpt-oss-120b',
            api_key=api_key,
            base_url='https://inference-api.alcf.anl.gov/resource_server/sophia/vllm/v1'
        )

        prompt = f"""You are a medicinal chemist AI.
Generate {n} small-molecule drug candidates targeting: {target_description}

Rules:
- Each molecule must be drug-like (Lipinski's Rule of Five)
- Prefer known pharmacophores for this target class
- Return ONLY valid JSON, no preamble, no markdown fences

Return a JSON array of objects with these fields:
  "smiles"      : valid SMILES string
  "name"        : short descriptive name
  "rationale"   : one sentence explaining why this scaffold is promising

Example format:
[
  {{"smiles": "CCO", "name": "ethanol", "rationale": "Simple example."}}
]
"""
        try:
            response = model.invoke(prompt)
            candidates = self._parse_json_response(response.content)
        except Exception as e:
            return f"Failed to connect to inference endpoint with {api_key}: error:{e}"
        
        return str(candidates)
        
    

In [3]:
UEP_CONFIG = {'account': "AuroraGPT", 'queue': 'debug', 
              # Polaris MEP is not set to user `worker_init`, but oddly `config_key` to set env
              'config_key': 'source /home/yadunand/setup_openmm.sh; which python3'}
UEP_CONFIG = None
# Local EP
EP_ID = "0a6b732b-71ff-43b8-9537-017ad80eaadb"
# Polaris MEP
# EP_ID = '9a947ba5-f537-4681-acf3-cc66485aadec'

async def main():

    init_logging(logging.INFO)
    
    with GlobusComputeExecutor(endpoint_id=EP_ID, 
                               user_endpoint_config=UEP_CONFIG
                              ) as remote_exec:
        # Create manager with agents and their assigned executors
        async with await Manager.from_exchange_factory(
            factory=HttpExchangeFactory(),
            executors=remote_exec,
        ) as manager:
            
            handle_generator = await manager.launch(CandidateGenerator)

            candidates = await handle_generator.generate_candidates("kinase inhibitor targeting EGFR for NSCLC")
            print(candidates)
        remote_exec.shutdown(wait=True)


await main()

[2026-04-04 11:37:18.372] INFO     (root) Configured logger (stdout-level=INFO, logfile=None, logfile-level=None)
[2026-04-04 11:37:18.668] INFO     (academy.exchange.cloud.client) Registered UserId<6b1c58d0> in exchange
[2026-04-04 11:37:18.674] INFO     (academy.manager) Initialized manager (UserId<6b1c58d0>)
[2026-04-04 11:37:18.726] INFO     (academy.exchange.client) Registered AgentId<c2968ca5> in exchange
[2026-04-04 11:37:18.733] INFO     (globus_compute_sdk.sdk.client) Registering function: _run_agent_on_worker
[2026-04-04 11:37:19.387] INFO     (academy.manager) Launched agent (AgentId<c2968ca5>; <class '__main__.CandidateGenerator'>)
[2026-04-04 11:37:19.388] INFO     (globus_compute_sdk.sdk.executor) Submitting tasks for Task Group None to Endpoint 0a6b732b-71ff-43b8-9537-017ad80eaadb: 1
[2026-04-04 11:37:20.666] INFO     (globus_compute_sdk.sdk.executor) Updating task_group_id from None to 404363aa-e204-4f3b-ac6d-dbf7e2c97fdc
[2026-04-04 11:37:21.059] INFO     (pika.adapter